[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_11/notebook_11_2_federated_learning.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 11.2: Federated Learning for Healthcare

**Volume 1: Foundations of AI in Healthcare**  
**Chapter 11: Privacy, Security, and Trustworthy AI**

---

## Introduction

Healthcare data is naturally distributed across multiple institutions—hospitals, clinics, research centers—each holding valuable patient records that could improve AI models if combined. However, sharing raw patient data faces significant barriers:

- **Privacy regulations**: HIPAA, GDPR, and local laws restrict data sharing
- **Institutional policies**: Hospitals are reluctant to share competitive advantages
- **Technical challenges**: Data transfer is expensive and time-consuming
- **Patient trust**: Concerns about data misuse and breaches

**Federated Learning (FL)** offers an elegant solution: train models collaboratively **without sharing raw data**. Instead of centralizing data, FL brings the model to the data:

1. Each institution trains a local model on its own data
2. Only model updates (gradients/weights) are shared with a central server
3. The server aggregates updates into a global model
4. The improved global model is sent back to institutions
5. Process repeats for multiple rounds

### Clinical Motivation: Multi-Hospital Sepsis Prediction

Sepsis is a leading cause of in-hospital death, and early-warning models need data from many sites: each hospital sees a different case mix (older or younger patients, different sepsis prevalence), and a model trained at one site often transfers poorly. Federated learning lets ten hospitals train one sepsis model from routine vitals and labs (lactate, heart rate, temperature, WBC, respiratory rate, systolic blood pressure, ...) **without sharing patient data**.

> **Side note: rare diseases.** FL matters even more when each site has only a handful of cases. For a rare genetic disorder with a prevalence of 1 in 100,000, a single hospital might see only 5–10 cases over several years, while ten hospitals together reach 50+ cases.

### Learning Objectives

By the end of this notebook, you will:

1. Understand the **FedAvg** (Federated Averaging) algorithm
2. Implement federated learning for multi-hospital sepsis prediction
3. Handle **non-IID** (non-Independent and Identically Distributed) data across institutions
4. Compare federated learning with centralized and local-only training
5. Understand privacy guarantees and limitations of FL
6. Explore secure aggregation protocols
7. Add **per-site differential privacy** and measure the privacy-utility trade-off over ε and local epochs

---

## Setup

In [1]:
# ── Setup (JupyterLite) ─────────────────────────────────────────────────────
import sys
if sys.platform == "emscripten":          # JupyterLite / Pyodide
    import micropip
    await micropip.install(["seaborn"])

In [2]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
np.random.seed(42)

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

Libraries imported successfully!
NumPy version: 2.4.6
Pandas version: 3.1.1


---

## Part 1: Federated Averaging (FedAvg) Algorithm

### Mathematical Foundation

The **FedAvg** algorithm (McMahan et al., 2017) is the foundational protocol for federated learning:

**Goal**: Minimize the global loss function:

$$
\min_{w} F(w) = \sum_{k=1}^{K} \frac{n_k}{n} F_k(w)
$$

Where:
- $K$ = number of participating institutions (hospitals)
- $n_k$ = number of data points at institution $k$
- $n = \sum_{k=1}^{K} n_k$ = total data points across all institutions
- $F_k(w)$ = local loss function at institution $k$
- $w$ = model parameters (weights)

### FedAvg Protocol

**Server side** (each global round $t$):
1. Initialize global model $w^{(t)}$
2. Sample a subset of institutions $S_t \subseteq \{1, ..., K\}$
3. Send $w^{(t)}$ to each selected institution
4. Receive updated models $w_k^{(t+1)}$ from each institution
5. Aggregate updates:

$$
w^{(t+1)} = \sum_{k \in S_t} \frac{n_k}{\sum_{j \in S_t} n_j} w_k^{(t+1)}
$$

**Client side** (institution $k$ in round $t$):
1. Receive global model $w^{(t)}$
2. Train locally for $E$ epochs on local data:

$$
w_k^{(t+1)} = w^{(t)} - \eta \nabla F_k(w^{(t)})
$$

3. Send updated model $w_k^{(t+1)}$ to server

### Key Properties

- **Privacy-preserving**: Raw data never leaves institutions
- **Communication-efficient**: Only model updates are transmitted
- **Weighted aggregation**: Larger datasets have proportionally more influence
- **Convergence guarantee**: Under certain conditions, FedAvg converges to a good solution

In [3]:
class FederatedLearning:
    """
    Federated Learning implementation using FedAvg algorithm.

    This class simulates a federated learning environment where multiple
    institutions (hospitals) collaborate to train a shared model without
    sharing their raw patient data.
    """

    def __init__(self, n_institutions: int, input_dim: int, learning_rate: float = 0.01):
        """
        Initialize federated learning system.

        Args:
            n_institutions: Number of participating hospitals
            input_dim: Number of input features
            learning_rate: Learning rate for local training
        """
        self.n_institutions = n_institutions
        self.input_dim = input_dim
        self.learning_rate = learning_rate

        # Initialize global model (logistic regression)
        self.global_weights = np.zeros(input_dim)
        self.global_bias = 0.0

        # Track metrics
        self.history = {
            'global_loss': [],
            'global_accuracy': [],
            'local_losses': [[] for _ in range(n_institutions)],
            'local_accuracies': [[] for _ in range(n_institutions)]
        }

    def sigmoid(self, z):
        """Sigmoid activation function."""
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def compute_loss(self, X, y, weights, bias):
        """
        Compute logistic loss (binary cross-entropy).

        L(w, b) = -1/n * Σ[y_i * log(ŷ_i) + (1-y_i) * log(1-ŷ_i)]
        """
        n = len(y)
        z = np.dot(X, weights) + bias
        predictions = self.sigmoid(z)

        # Prevent log(0)
        predictions = np.clip(predictions, 1e-10, 1 - 1e-10)

        loss = -np.mean(y * np.log(predictions) + (1 - y) * np.log(1 - predictions))
        return loss

    def compute_gradients(self, X, y, weights, bias):
        """
        Compute gradients for logistic regression.

        ∇w = 1/n * X^T * (ŷ - y)
        ∇b = 1/n * Σ(ŷ - y)
        """
        n = len(y)
        z = np.dot(X, weights) + bias
        predictions = self.sigmoid(z)

        errors = predictions - y
        grad_w = np.dot(X.T, errors) / n
        grad_b = np.mean(errors)

        return grad_w, grad_b

    def local_train(self, X, y, initial_weights, initial_bias, epochs=5):
        """
        Train model locally at one institution.

        Args:
            X: Local training features
            y: Local training labels
            initial_weights: Starting weights from global model
            initial_bias: Starting bias from global model
            epochs: Number of local training epochs

        Returns:
            Updated weights, bias, and metrics
        """
        weights = initial_weights.copy()
        bias = initial_bias

        for epoch in range(epochs):
            # Compute gradients
            grad_w, grad_b = self.compute_gradients(X, y, weights, bias)

            # Update parameters
            weights -= self.learning_rate * grad_w
            bias -= self.learning_rate * grad_b

        # Compute final loss and accuracy
        loss = self.compute_loss(X, y, weights, bias)
        predictions = (self.sigmoid(np.dot(X, weights) + bias) >= 0.5).astype(int)
        accuracy = np.mean(predictions == y)

        return weights, bias, loss, accuracy

    def aggregate_models(self, local_updates: List[Tuple], data_sizes: List[int]):
        """
        Aggregate local model updates using weighted averaging.

        FedAvg aggregation:
        w_global = Σ (n_k / n_total) * w_k

        Args:
            local_updates: List of (weights, bias) tuples from each institution
            data_sizes: Number of training samples at each institution

        Returns:
            Aggregated global weights and bias
        """
        total_samples = sum(data_sizes)

        # Weighted averaging
        aggregated_weights = np.zeros_like(self.global_weights)
        aggregated_bias = 0.0

        for (weights, bias), n_samples in zip(local_updates, data_sizes):
            weight_factor = n_samples / total_samples
            aggregated_weights += weight_factor * weights
            aggregated_bias += weight_factor * bias

        return aggregated_weights, aggregated_bias

    def train_federated(self, institutions_data: List[Tuple],
                       global_test_X, global_test_y,
                       n_rounds: int = 20, local_epochs: int = 5):
        """
        Execute federated learning for multiple rounds.

        Args:
            institutions_data: List of (X_train, y_train) for each institution
            global_test_X: Global test features for evaluation
            global_test_y: Global test labels for evaluation
            n_rounds: Number of federated learning rounds
            local_epochs: Epochs of local training per round
        """
        print(f"Starting Federated Learning with {self.n_institutions} institutions...")
        print(f"Rounds: {n_rounds}, Local epochs per round: {local_epochs}\n")

        for round_num in range(n_rounds):
            print(f"Round {round_num + 1}/{n_rounds}")

            # Step 1: Each institution trains locally
            local_updates = []
            data_sizes = []

            for inst_id, (X_train, y_train) in enumerate(institutions_data):
                # Local training
                weights, bias, loss, accuracy = self.local_train(
                    X_train, y_train,
                    self.global_weights, self.global_bias,
                    epochs=local_epochs
                )

                local_updates.append((weights, bias))
                data_sizes.append(len(y_train))

                # Track local metrics
                self.history['local_losses'][inst_id].append(loss)
                self.history['local_accuracies'][inst_id].append(accuracy)

                print(f"  Institution {inst_id + 1}: Loss={loss:.4f}, Accuracy={accuracy:.4f}")

            # Step 2: Server aggregates updates
            self.global_weights, self.global_bias = self.aggregate_models(
                local_updates, data_sizes
            )

            # Step 3: Evaluate global model
            global_loss = self.compute_loss(
                global_test_X, global_test_y,
                self.global_weights, self.global_bias
            )
            global_preds = (self.sigmoid(
                np.dot(global_test_X, self.global_weights) + self.global_bias
            ) >= 0.5).astype(int)
            global_accuracy = np.mean(global_preds == global_test_y)

            self.history['global_loss'].append(global_loss)
            self.history['global_accuracy'].append(global_accuracy)

            print(f"  Global Model: Loss={global_loss:.4f}, Accuracy={global_accuracy:.4f}\n")

        print("Federated Learning completed!")

    def predict(self, X):
        """Make predictions using the global model."""
        z = np.dot(X, self.global_weights) + self.global_bias
        return (self.sigmoid(z) >= 0.5).astype(int)

    def predict_proba(self, X):
        """Return prediction probabilities."""
        z = np.dot(X, self.global_weights) + self.global_bias
        return self.sigmoid(z)

print("FederatedLearning class defined successfully!")

FederatedLearning class defined successfully!


---

## Part 2: Simulating Multi-Hospital Collaboration

### Scenario: Multi-Hospital Sepsis Prediction (synthetic data)

We simulate 10 hospitals collaborating to predict sepsis in admitted patients. Each hospital has:

- **Different patient populations** (non-IID data: different age mix and sepsis prevalence, roughly 5–25%)
- **Varying dataset sizes** (roughly 60–400 patients each)
- **Local privacy constraints** (cannot share raw data)

**Features** (synthetic vitals and labs at the time of assessment): age, heart rate, respiratory rate, temperature, systolic blood pressure (SBP), white blood cell count (WBC), lactate, creatinine.

**Target**: Binary label (0 = no sepsis, 1 = sepsis)

> ⚠️ All data here are **synthetic**. Septic patients are drawn with higher lactate, HR, RR, temperature, WBC and creatinine and lower SBP; the effect sizes are made deliberately modest so that small hospitals struggle on their own.


In [4]:
FEATURES = ['age', 'heart_rate', 'resp_rate', 'temperature', 'sbp', 'wbc', 'lactate', 'creatinine']
# Pre-agreed reference values used by every site to standardise features
# (published clinical ranges, so no patient data has to be pooled to compute them)
REF_MEAN = np.array([60, 90, 18, 37.2, 120, 9.5, 1.8, 1.1])
REF_SD   = np.array([16, 18,  5,  0.9,  22, 4.0, 1.2, 0.5])

def make_sepsis_patients(n, prevalence, age_mean, rng):
    """Synthetic admissions: sepsis status first, then vitals/labs that depend on it.
    The feature-given-label relationship is the SAME at every hospital; prevalence and age mix differ."""
    s = (rng.random(n) < prevalence).astype(int)          # 1 = sepsis
    X = np.column_stack([
        rng.normal(age_mean + 3*s, 15, n),                  # age (years)
        rng.normal(85 + 7*s, 15, n),                        # heart rate (bpm)
        rng.normal(17 + 2*s, 4, n),                         # respiratory rate (/min)
        rng.normal(37.0 + 0.3*s, 0.7, n),                   # temperature (°C)
        rng.normal(125 - 7*s, 20, n),                       # systolic BP (mmHg)
        rng.normal(8.5 + 1.6*s, 3.5, n),                    # WBC (×10⁹/L)
        rng.lognormal(np.log(1.3) + 0.3*s, 0.4, n),         # lactate (mmol/L)
        rng.lognormal(np.log(0.95) + 0.12*s, 0.3, n),       # creatinine (mg/dL)
    ])
    return X, s

def generate_hospital_data(n_hospitals=10, seed=42):
    """
    Generate synthetic sepsis data for multiple hospitals (non-IID):
    - different dataset sizes (60-400 patients)
    - different sepsis prevalence (5-25%)
    - different age mix (mean age 50-72)

    Returns:
        institutions_data: List of (X_train, y_train) for each hospital
        global_test_data: pooled 20% hold-out from every hospital
        feature_names: Names of features
    """
    rng = np.random.default_rng(seed)
    institutions_data, all_test_X, all_test_y = [], [], []

    print("Generating data for hospitals...\n")
    for hospital_id in range(n_hospitals):
        n_samples = int(rng.integers(60, 400))
        prevalence = rng.uniform(0.05, 0.25)
        age_mean = rng.uniform(50, 72)

        X, y = make_sepsis_patients(n_samples, prevalence, age_mean, rng)
        X = (X - REF_MEAN) / REF_SD

        # Split into train/test (80/20 for each hospital)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=hospital_id, stratify=y
        )

        institutions_data.append((X_train, y_train))
        all_test_X.append(X_test)
        all_test_y.append(y_test)

        print(f"Hospital {hospital_id + 1} (mean age {age_mean:.0f}):")
        print(f"  Training samples: {len(y_train)} (Sepsis: {y_train.sum()}, {y_train.mean():.2%})")
        print(f"  Test samples: {len(y_test)} (Sepsis: {y_test.sum()}, {y_test.mean():.2%})")

    # Combine all test sets for global evaluation
    global_test_X = np.vstack(all_test_X)
    global_test_y = np.concatenate(all_test_y)

    print(f"\nGlobal test set: {len(global_test_y)} samples")
    print(f"Overall sepsis prevalence: {global_test_y.mean():.2%}")

    return institutions_data, (global_test_X, global_test_y), FEATURES

# Generate data
institutions_data, (global_test_X, global_test_y), feature_names = generate_hospital_data(n_hospitals=10)


Generating data for hospitals...

Hospital 1 (mean age 69):
  Training samples: 72 (Sepsis: 9, 12.50%)
  Test samples: 18 (Sepsis: 2, 11.11%)
Hospital 2 (mean age 61):
  Training samples: 258 (Sepsis: 56, 21.71%)
  Test samples: 65 (Sepsis: 14, 21.54%)
Hospital 3 (mean age 54):
  Training samples: 121 (Sepsis: 12, 9.92%)
  Test samples: 31 (Sepsis: 3, 9.68%)
Hospital 4 (mean age 50):
  Training samples: 104 (Sepsis: 8, 7.69%)
  Test samples: 26 (Sepsis: 2, 7.69%)
Hospital 5 (mean age 62):
  Training samples: 191 (Sepsis: 45, 23.56%)
  Test samples: 48 (Sepsis: 11, 22.92%)
Hospital 6 (mean age 65):
  Training samples: 165 (Sepsis: 46, 27.88%)
  Test samples: 42 (Sepsis: 12, 28.57%)
Hospital 7 (mean age 52):
  Training samples: 188 (Sepsis: 25, 13.30%)
  Test samples: 47 (Sepsis: 6, 12.77%)
Hospital 8 (mean age 51):
  Training samples: 253 (Sepsis: 34, 13.44%)
  Test samples: 64 (Sepsis: 8, 12.50%)
Hospital 9 (mean age 51):
  Training samples: 131 (Sepsis: 24, 18.32%)
  Test samples: 33 

### Visualizing Data Distribution Across Hospitals

In [5]:
# Visualize data distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Dataset sizes
train_sizes = [len(y) for X, y in institutions_data]
axes[0].bar(range(1, 11), train_sizes, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Hospital ID', fontsize=12)
axes[0].set_ylabel('Number of Training Samples', fontsize=12)
axes[0].set_title('Training Data Distribution Across Hospitals', fontsize=14, fontweight='bold')
axes[0].set_xticks(range(1, 11))
axes[0].grid(axis='y', alpha=0.3)

# Prevalence rates
prevalence_rates = [y.mean() for X, y in institutions_data]
axes[1].bar(range(1, 11), prevalence_rates, color='coral', alpha=0.7)
axes[1].axhline(y=np.mean(prevalence_rates), color='red', linestyle='--',
                label=f'Mean: {np.mean(prevalence_rates):.2%}')
axes[1].set_xlabel('Hospital ID', fontsize=12)
axes[1].set_ylabel('Sepsis Prevalence', fontsize=12)
axes[1].set_title('Non-IID Data: Varying Prevalence Rates', fontsize=14, fontweight='bold')
axes[1].set_xticks(range(1, 11))
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Data Characteristics:")
print(f"Total training samples: {sum(train_sizes)}")
print(f"Average samples per hospital: {np.mean(train_sizes):.1f} ± {np.std(train_sizes):.1f}")
print(f"Prevalence range: {min(prevalence_rates):.2%} - {max(prevalence_rates):.2%}")
print("\n⚠️ Non-IID Challenge: Different hospitals have different data distributions!")


📊 Data Characteristics:
Total training samples: 1662
Average samples per hospital: 166.2 ± 57.7
Prevalence range: 7.69% - 27.88%

⚠️ Non-IID Challenge: Different hospitals have different data distributions!


---

## Part 3: Training Federated Model

Now we train a federated model across all 10 hospitals using the FedAvg algorithm.

In [6]:
# Initialize federated learning system
fl_system = FederatedLearning(
    n_institutions=10,
    input_dim=len(feature_names),
    learning_rate=0.1
)

# Train federated model
fl_system.train_federated(
    institutions_data=institutions_data,
    global_test_X=global_test_X,
    global_test_y=global_test_y,
    n_rounds=15,
    local_epochs=5
)

Starting Federated Learning with 10 institutions...
Rounds: 15, Local epochs per round: 5

Round 1/15
  Institution 1: Loss=0.5785, Accuracy=0.8611
  Institution 2: Loss=0.6065, Accuracy=0.8333
  Institution 3: Loss=0.5619, Accuracy=0.9091
  Institution 4: Loss=0.5641, Accuracy=0.9135
  Institution 5: Loss=0.6260, Accuracy=0.7958
  Institution 6: Loss=0.6312, Accuracy=0.7879
  Institution 7: Loss=0.5776, Accuracy=0.8830
  Institution 8: Loss=0.5828, Accuracy=0.8656
  Institution 9: Loss=0.6090, Accuracy=0.8244
  Institution 10: Loss=0.5988, Accuracy=0.8268
  Global Model: Loss=0.6078, Accuracy=0.8425

Round 2/15
  Institution 1: Loss=0.5190, Accuracy=0.8750
  Institution 2: Loss=0.5498, Accuracy=0.8295
  Institution 3: Loss=0.4960, Accuracy=0.9174
  Institution 4: Loss=0.4971, Accuracy=0.9231
  Institution 5: Loss=0.5736, Accuracy=0.8063
  Institution 6: Loss=0.5858, Accuracy=0.7636
  Institution 7: Loss=0.5146, Accuracy=0.8883
  Institution 8: Loss=0.5232, Accuracy=0.8735
  Institutio

  Institution 4: Loss=0.3858, Accuracy=0.9327
  Institution 5: Loss=0.4916, Accuracy=0.8010
  Institution 6: Loss=0.5175, Accuracy=0.7455
  Institution 7: Loss=0.4122, Accuracy=0.8936
  Institution 8: Loss=0.4261, Accuracy=0.8735
  Institution 9: Loss=0.4791, Accuracy=0.8244
  Institution 10: Loss=0.4553, Accuracy=0.8547
  Global Model: Loss=0.4551, Accuracy=0.8425

Round 6/15
  Institution 1: Loss=0.4033, Accuracy=0.8750
  Institution 2: Loss=0.4436, Accuracy=0.8140
  Institution 3: Loss=0.3672, Accuracy=0.9091
  Institution 4: Loss=0.3647, Accuracy=0.9327
  Institution 5: Loss=0.4772, Accuracy=0.8010
  Institution 6: Loss=0.5062, Accuracy=0.7455
  Institution 7: Loss=0.3933, Accuracy=0.8936
  Institution 8: Loss=0.4080, Accuracy=0.8735
  Institution 9: Loss=0.4658, Accuracy=0.8244
  Institution 10: Loss=0.4379, Accuracy=0.8547
  Global Model: Loss=0.4380, Accuracy=0.8425

Round 7/15
  Institution 1: Loss=0.3890, Accuracy=0.8750
  Institution 2: Loss=0.4310, Accuracy=0.8140
  Institut

### Visualizing Federated Learning Progress

In [7]:
# Plot training progress
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Global model performance
rounds = range(1, len(fl_system.history['global_loss']) + 1)
axes[0].plot(rounds, fl_system.history['global_loss'], marker='o', linewidth=2,
            markersize=6, color='darkblue', label='Global Loss')
axes[0].set_xlabel('Federated Round', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Global Model: Training Loss', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(rounds, fl_system.history['global_accuracy'], marker='o', linewidth=2,
            markersize=6, color='darkgreen', label='Global Accuracy')
axes[1].set_xlabel('Federated Round', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Global Model: Test Accuracy', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

# Final performance
final_accuracy = fl_system.history['global_accuracy'][-1]
final_loss = fl_system.history['global_loss'][-1]

print("\n🎯 Final Federated Model Performance:")
print(f"Accuracy: {final_accuracy:.4f}")
print(f"Loss: {final_loss:.4f}")


🎯 Final Federated Model Performance:
Accuracy: 0.8401
Loss: 0.3774


---

## Part 4: Comparison with Baseline Approaches

To demonstrate the value of federated learning, we compare three approaches:

1. **Local-only training**: Each hospital trains independently on its own data
2. **Centralized training**: All data pooled together (ideal but privacy-violating)
3. **Federated learning**: Our FedAvg approach (privacy-preserving collaboration)

Expected pattern (we check below whether it holds in this run):
- **Local-only** suffers from small dataset sizes
- **Centralized** achieves best performance but violates privacy
- **Federated** approaches centralized performance while preserving privacy

In [8]:
# 1. Local-only training (each hospital trains independently)
from sklearn.metrics import recall_score
print("Training local-only models...\n")
local_accuracies = []
local_sens, local_aucs = [], []

for hospital_id, (X_train, y_train) in enumerate(institutions_data):
    # Train sklearn model locally
    local_model = LogisticRegression(max_iter=1000, random_state=42)
    local_model.fit(X_train, y_train)

    # Evaluate on global test set
    accuracy = local_model.score(global_test_X, global_test_y)
    local_accuracies.append(accuracy)
    local_sens.append(recall_score(global_test_y, local_model.predict(global_test_X)))
    local_aucs.append(roc_auc_score(global_test_y, local_model.predict_proba(global_test_X)[:, 1]))
    print(f"Hospital {hospital_id + 1} - Local model accuracy: {accuracy:.4f}")

avg_local_accuracy = np.mean(local_accuracies)
print(f"\nAverage local-only accuracy: {avg_local_accuracy:.4f} ± {np.std(local_accuracies):.4f}")

# 2. Centralized training (pool all data - ideal but privacy-violating)
print("\n" + "="*60)
print("Training centralized model (privacy-violating baseline)...\n")

# Combine all training data
all_X_train = np.vstack([X for X, y in institutions_data])
all_y_train = np.concatenate([y for X, y in institutions_data])

centralized_model = LogisticRegression(max_iter=1000, random_state=42)
centralized_model.fit(all_X_train, all_y_train)
centralized_accuracy = centralized_model.score(global_test_X, global_test_y)

print(f"Centralized model accuracy: {centralized_accuracy:.4f}")

# 3. Federated learning (our approach)
federated_accuracy = fl_system.history['global_accuracy'][-1]

print("\n" + "="*60)
print("\n📊 COMPARISON SUMMARY\n")
federated_sens = recall_score(global_test_y, fl_system.predict(global_test_X))
federated_auc = roc_auc_score(global_test_y, fl_system.predict_proba(global_test_X))
centralized_sens = recall_score(global_test_y, centralized_model.predict(global_test_X))
centralized_auc = roc_auc_score(global_test_y, centralized_model.predict_proba(global_test_X)[:, 1])
majority_accuracy = 1 - global_test_y.mean()
print(f"{'Approach':<24} {'Accuracy':<10} {'Sensitivity':<12} {'AUC':<8} {'Privacy-Preserving'}")
print("-" * 72)
print(f"{'Always predict negative':<24} {majority_accuracy:<10.4f} {0.0:<12.4f} {0.5:<8.3f} {'—'}")
print(f"{'Local-only (avg)':<24} {avg_local_accuracy:<10.4f} {np.mean(local_sens):<12.4f} {np.mean(local_aucs):<8.3f} {'✓'}")
print(f"{'Federated Learning':<24} {federated_accuracy:<10.4f} {federated_sens:<12.4f} {federated_auc:<8.3f} {'✓'}")
print(f"{'Centralized (ideal)':<24} {centralized_accuracy:<10.4f} {centralized_sens:<12.4f} {centralized_auc:<8.3f} {'✗'}")
print("="*72)

# Calculate improvement
improvement_vs_local = ((federated_accuracy - avg_local_accuracy) / avg_local_accuracy) * 100
gap_vs_centralized = centralized_accuracy - federated_accuracy

print(f"\n📊 What the numbers say (this run, {int(global_test_y.sum())} positives in {len(global_test_y)} test patients):")
print(f"  • Accuracy: federated {improvement_vs_local:+.1f}% vs local-only, {gap_vs_centralized:+.4f} gap to centralized.")
print(f"    But 'always predict negative' already scores {majority_accuracy:.4f}, so accuracy says little here.")
print(f"  • AUC (ranking): local-only {np.mean(local_aucs):.3f} → federated {federated_auc:.3f} → centralized {centralized_auc:.3f}")
print(f"  • Sensitivity at the 0.5 threshold: federated catches {federated_sens:.1%} of cases "
      f"(local-only avg {np.mean(local_sens):.1%}, centralized {centralized_sens:.1%})")
if federated_auc > np.mean(local_aucs) and federated_sens < 0.5:
    print("  → Collaboration improved ranking (AUC), but at the default 0.5 cut-off the federated model")
    print(f"    still misses most cases: with {global_test_y.mean():.0%} prevalence and only {len(fl_system.history['global_loss'])} rounds the predicted risks stay low.")
    print("    Exercise: pick the threshold on validation data (e.g., for 80% sensitivity) and re-compare.")
elif federated_auc > np.mean(local_aucs) and federated_sens > np.mean(local_sens):
    print("  → Collaboration improved both ranking (AUC) and case finding over local-only models.")
elif federated_auc > np.mean(local_aucs):
    print("  → Collaboration improved ranking (AUC), but not sensitivity at the 0.5 cut-off versus local models.")
else:
    print("  → In this run federated learning did NOT beat local-only models on AUC.")
print(f"  • Raw data stayed at all {len(institutions_data)} hospitals (model updates can still leak information; see Part 5)")

Training local-only models...

Hospital 1 - Local model accuracy: 0.8258
Hospital 2 - Local model accuracy: 0.8544
Hospital 3 - Local model accuracy: 0.8329
Hospital 4 - Local model accuracy: 0.8377
Hospital 5 - Local model accuracy: 0.8425


Hospital 6 - Local model accuracy: 0.8210
Hospital 7 - Local model accuracy: 0.8496
Hospital 8 - Local model accuracy: 0.8401


Hospital 9 - Local model accuracy: 0.8353
Hospital 10 - Local model accuracy: 0.8401

Average local-only accuracy: 0.8379 ± 0.0095

Training centralized model (privacy-violating baseline)...

Centralized model accuracy: 0.8473


📊 COMPARISON SUMMARY

Approach                 Accuracy   Sensitivity  AUC      Privacy-Preserving
------------------------------------------------------------------------
Always predict negative  0.8305     0.0000       0.500    —
Local-only (avg)         0.8379     0.2394       0.804    ✓
Federated Learning       0.8401     0.1690       0.830    ✓
Centralized (ideal)      0.8473     0.2676       0.834    ✗

📊 What the numbers say (this run, 71 positives in 419 test patients):
  • Accuracy: federated +0.3% vs local-only, +0.0072 gap to centralized.
    But 'always predict negative' already scores 0.8305, so accuracy says little here.
  • AUC (ranking): local-only 0.804 → federated 0.830 → centralized 0.834
  • Sensitivity at the 0.5 threshold: federated catche

In [9]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart comparison
approaches = ['Local-only\n(avg)', 'Federated\nLearning', 'Centralized\n(ideal)']
accuracies = [avg_local_accuracy, federated_accuracy, centralized_accuracy]
colors = ['coral', 'steelblue', 'lightgray']

bars = axes[0].bar(approaches, accuracies, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylim([0.0, 1.05])
axes[0].axhline(majority_accuracy, color='gray', linestyle='--', label=f'Always negative ({majority_accuracy:.3f})')
axes[0].legend(loc='lower right')
axes[0].grid(axis='y', alpha=0.3)

# Add value labels
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height,
                f'{acc:.4f}',
                ha='center', va='bottom', fontsize=11, fontweight='bold')

# Privacy vs Performance trade-off
privacy_scores = [1.0, 1.0, 0.0]  # 1 = privacy-preserving, 0 = not

axes[1].scatter([privacy_scores[0]], [accuracies[0]], s=200, color=colors[0],
               alpha=0.7, edgecolors='black', linewidth=2, label='Local-only')
axes[1].scatter([privacy_scores[1]], [accuracies[1]], s=200, color=colors[1],
               alpha=0.7, edgecolors='black', linewidth=2, label='Federated')
axes[1].scatter([privacy_scores[2]], [accuracies[2]], s=200, color=colors[2],
               alpha=0.7, edgecolors='black', linewidth=2, label='Centralized')

axes[1].set_xlabel('Privacy Preservation', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Privacy-Performance Trade-off', fontsize=14, fontweight='bold')
axes[1].set_xlim([-0.2, 1.2])
axes[1].set_ylim([0.0, 1.05])
axes[1].axhline(majority_accuracy, color='gray', linestyle='--')
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['No Privacy', 'Full Privacy'])
axes[1].legend(loc='lower left')
axes[1].grid(alpha=0.3)

# Annotate the sweet spot
axes[1].annotate(f'Federated: accuracy {accuracies[1]:.3f},\nbut sensitivity only {federated_sens:.1%}',
                xy=(privacy_scores[1], accuracies[1]),
                xytext=(0.3, 0.45),
                arrowprops=dict(arrowstyle='->', lw=2, color='green'),
                fontsize=11, color='green', fontweight='bold')

plt.tight_layout()
plt.show()

---

## Part 5: Privacy Analysis and Secure Aggregation

### Privacy Guarantees and Limitations

While federated learning prevents raw data sharing, it still has privacy vulnerabilities:

**Threats**:
1. **Gradient leakage**: Model updates can leak information about training data
2. **Model inversion**: Adversaries may reconstruct training samples from model weights
3. **Membership inference**: Attackers can determine if a specific patient was in training data

**Defense: Secure Aggregation**

Secure aggregation protocols (Bonawitz et al., 2017) ensure the server learns only the aggregated model, not individual updates:

$$
w_{\text{global}} = \sum_{k=1}^{K} w_k
$$

But the server cannot access any individual $w_k$.

**Implementation using Secret Sharing**:

1. Each institution $k$ splits its update $w_k$ into shares:
   $$w_k = s_{k,1} + s_{k,2} + ... + s_{k,K}$$

2. Institution $k$ sends share $s_{k,j}$ to institution $j$

3. Each institution sums received shares: $S_j = \sum_{k=1}^{K} s_{k,j}$

4. Server aggregates: $w_{\text{global}} = \sum_{j=1}^{K} S_j = \sum_{k=1}^{K} w_k$

**Combined with Differential Privacy**:

For strongest protection, combine FL with DP:
- Add noise to gradients before sharing (DP-SGD) — implemented below as per-site DP
- Use secure aggregation to hide individual contributions
- Result: Formal privacy guarantee + protection against model inversion

In [10]:
def simple_secure_aggregation_demo(local_updates, n_institutions):
    """
    Simplified demonstration of secure aggregation concept.

    In reality, this uses cryptographic protocols like:
    - Secret sharing (Shamir's scheme)
    - Masking with random values
    - Secure multi-party computation

    This demo shows the concept without full cryptographic implementation.
    """
    print("🔒 Secure Aggregation Protocol Demo\n")
    print("Simulating secure aggregation where server cannot see individual updates...\n")

    # Step 1: Each institution creates random masks
    print("Step 1: Institutions generate pairwise random masks")
    masks = {}
    for i in range(n_institutions):
        for j in range(i + 1, n_institutions):
            # Pairwise random mask (same for both institutions)
            mask = np.random.randn(*local_updates[0].shape) * 0.1
            masks[(i, j)] = mask

    # Step 2: Each institution masks its update
    print("Step 2: Institutions add/subtract masks to their updates")
    masked_updates = []
    for i in range(n_institutions):
        masked = local_updates[i].copy()
        # Add masks with institutions j > i, subtract masks with j < i
        for j in range(n_institutions):
            if i < j:
                masked += masks[(i, j)]
            elif i > j:
                masked -= masks[(j, i)]
        masked_updates.append(masked)

    print("Step 3: Server receives masked updates (individual updates are hidden)")
    print("Step 4: Server aggregates masked updates\n")

    # Step 3: Server aggregates (masks cancel out)
    secure_aggregate = sum(masked_updates) / n_institutions

    # Step 4: Compare with direct aggregation
    direct_aggregate = sum(local_updates) / n_institutions

    # Verify they're the same (masks cancelled)
    difference = np.linalg.norm(secure_aggregate - direct_aggregate)

    print("✅ Results:")
    print(f"  • Server computed correct aggregate: {difference < 1e-10}")
    print(f"  • Server only saw masked vectors (pairwise masks cancel in the sum)")
    print(f"  • Caveat: in this toy demo the masks are small (σ=0.1) and generated centrally;")
    print(f"    real protocols derive them from key agreement and handle dropouts")
    print(f"  • Difference (should be ~0): {difference:.2e}")

    return secure_aggregate

# Demo secure aggregation
print("=" * 70)
# Each hospital's update = its locally trained weights (one local round from the current global model)
example_updates = [fl_system.local_train(X, y, fl_system.global_weights, fl_system.global_bias, epochs=1)[0]
                   for X, y in institutions_data]
secure_result = simple_secure_aggregation_demo(example_updates, n_institutions=10)
print("=" * 70)

🔒 Secure Aggregation Protocol Demo

Simulating secure aggregation where server cannot see individual updates...

Step 1: Institutions generate pairwise random masks
Step 2: Institutions add/subtract masks to their updates
Step 3: Server receives masked updates (individual updates are hidden)
Step 4: Server aggregates masked updates

✅ Results:
  • Server computed correct aggregate: True
  • Server only saw masked vectors (pairwise masks cancel in the sum)
  • Caveat: in this toy demo the masks are small (σ=0.1) and generated centrally;
    real protocols derive them from key agreement and handle dropouts
  • Difference (should be ~0): 6.21e-17


### Part 5b: Per-Site Differential Privacy (DP-FedAvg)

Secure aggregation hides individual updates from the server, but the updates and the final model can still leak patient information. Here every hospital trains with **differentially private gradient descent on its own data**:

1. compute each patient's gradient and **clip** it to norm ≤ C,
2. add **Gaussian noise** with standard deviation z·C to the summed gradient,
3. divide by a **fixed, public normaliser** N̄ (not the site's true size, which itself depends on who is in the data) and take the step; after E local epochs send the (already private) update to the server, which averages the sites with **equal, public weights**.

**Accounting.** Each hospital runs T rounds × E epochs = T·E full-batch Gaussian mechanisms on its own patients (**add/remove-one-patient adjacency**: adding or removing one patient changes the clipped gradient *sum* by at most C, and because N̄ and the aggregation weights are public constants nothing else changes). We compose them with zero-concentrated DP: each step costs ρ = 1/(2z²), and a total ρ gives (ε, δ)-DP with ε = ρ + 2√(ρ ln(1/δ)). For a target ε we solve for the noise multiplier z. Since hospitals hold disjoint patients, the guarantee holds for every patient at every site, and nothing the server does afterwards can weaken it (post-processing).

Note the tension: **more local epochs E means more noisy steps to pay for**, so for a fixed ε each step must be noisier. We sweep ε ∈ {0.5, 1, 3, 10} and E ∈ {1, 5, 10} (plus no-DP), averaging the test AUC over 5 random seeds.

In [11]:
N_PUBLIC = 200   # pre-agreed nominal site size, used as the gradient normaliser at every site

def dp_fedavg(institutions_data, test_X, test_y, epsilon, local_epochs,
              n_rounds=15, clip=1.0, delta=1e-5, lr=0.5, seed=0, n_public=N_PUBLIC):
    """FedAvg in which every hospital runs DP gradient descent (per-patient clipping + Gaussian noise).
    epsilon=np.inf turns DP off. Returns (test AUC, noise multiplier z)."""
    rng = np.random.default_rng(seed)
    d = test_X.shape[1]
    w = np.zeros(d + 1)                                   # weights + bias
    steps = n_rounds * local_epochs                       # noisy steps each hospital pays for
    if np.isfinite(epsilon):
        L = np.log(1 / delta)
        rho = (np.sqrt(L + epsilon) - np.sqrt(L)) ** 2   # zCDP budget that yields (ε, δ)
        z = np.sqrt(steps / (2 * rho))                    # noise multiplier per step
    else:
        z = 0.0
    for _ in range(n_rounds):
        updates = []
        for X, y in institutions_data:
            Xb = np.hstack([X, np.ones((len(X), 1))])
            wl = w.copy()
            for _ in range(local_epochs):
                g = (fl_system.sigmoid(Xb @ wl) - y)[:, None] * Xb      # per-patient gradients
                if z > 0:
                    norms = np.maximum(np.linalg.norm(g, axis=1), 1e-12)
                    g = g * np.minimum(1.0, clip / norms)[:, None]       # clip each patient
                    g_sum = g.sum(axis=0) + rng.normal(0, z * clip, d + 1)  # Gaussian noise
                else:
                    g_sum = g.sum(axis=0)
                wl -= lr * g_sum / n_public   # public constant: the true site size never enters the update
            updates.append(wl - w)
        w = w + np.mean(updates, axis=0)            # equal public weights (true sizes would leak)
    return roc_auc_score(test_y, fl_system.sigmoid(test_X @ w[:-1] + w[-1])), z

eps_grid = [0.5, 1, 3, 10, np.inf]
epoch_grid = [1, 5, 10]
rows = []
for E in epoch_grid:
    for eps in eps_grid:
        for seed in range(5):
            auc_s, z = dp_fedavg(institutions_data, global_test_X, global_test_y, eps, E, seed=seed)
            rows.append({'E': E, 'epsilon': eps, 'seed': seed, 'AUC': auc_s, 'z': z})
dp_df = pd.DataFrame(rows)
dp_summary = dp_df.groupby(['E', 'epsilon']).agg(AUC_mean=('AUC', 'mean'), AUC_sd=('AUC', 'std'),
                                                  noise_z=('z', 'first')).reset_index()

print("DP-FedAvg: mean test AUC over 5 seeds (rows = local epochs E, columns = ε; inf = no DP)\n")
print(dp_summary.pivot(index='E', columns='epsilon', values='AUC_mean').round(3).to_string())
print("\nNoise multiplier z per step (larger = noisier):\n")
print(dp_summary.pivot(index='E', columns='epsilon', values='noise_z').round(1).to_string())


DP-FedAvg: mean test AUC over 5 seeds (rows = local epochs E, columns = ε; inf = no DP)

epsilon   0.5    1.0    3.0    10.0   inf 
E                                         
1        0.821  0.829  0.830  0.830  0.829
5        0.780  0.816  0.831  0.834  0.834
10       0.757  0.806  0.831  0.835  0.835

Noise multiplier z per step (larger = noisier):

epsilon   0.5   1.0   3.0   10.0  inf 
E                                     
1         37.6  19.0   6.6   2.2   0.0
5         84.0  42.4  14.7   4.9   0.0
10       118.8  60.0  20.8   7.0   0.0


In [12]:
fig, ax = plt.subplots(figsize=(9, 5))
finite = dp_summary[np.isfinite(dp_summary['epsilon'])]
for E in epoch_grid:
    sub = finite[finite['E'] == E]
    ax.errorbar(sub['epsilon'], sub['AUC_mean'], yerr=sub['AUC_sd'], marker='o', capsize=3, label=f'E = {E} local epochs')
ax.axhline(centralized_auc, color='gray', linestyle='--', label=f'Centralized, no DP ({centralized_auc:.3f})')
ax.axhline(np.mean(local_aucs), color='coral', linestyle=':', label=f'Local-only avg, no DP ({np.mean(local_aucs):.3f})')
ax.set_xscale('log')
ax.set_xticks([0.5, 1, 3, 10]); ax.set_xticklabels(['0.5', '1', '3', '10'])
ax.set_xlabel('Privacy budget ε per hospital (δ = 1e-5)', fontsize=12)
ax.set_ylabel('Test AUC (mean ± SD, 5 seeds)', fontsize=12)
ax.set_title('DP-FedAvg: Privacy-Utility Trade-off for Sepsis Prediction', fontsize=14, fontweight='bold')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Conclusions computed from the sweep
m = dp_summary.set_index(['E', 'epsilon'])['AUC_mean']
local_avg = np.mean(local_aucs)
print("💡 What the sweep shows (this run):")
for E in epoch_grid:
    drop = m[(E, np.inf)] - m[(E, 0.5)]
    beats = [e for e in [0.5, 1, 3, 10] if m[(E, e)] > local_avg]
    msg = f"from ε={min(beats):g} upward" if beats else "at none of the tested ε"
    print(f"  • E={E:>2}: AUC {m[(E, np.inf)]:.3f} without DP → {m[(E, 0.5)]:.3f} at ε=0.5 (loss {drop:.3f}); "
          f"beats the local-only average ({local_avg:.3f}) {msg}")
best_strong = max(epoch_grid, key=lambda E: m[(E, 0.5)])
best_weak = max(epoch_grid, key=lambda E: m[(E, 10)])
if best_strong != best_weak:
    print(f"  • The best number of local epochs depends on ε: E={best_strong} at ε=0.5 but E={best_weak} at ε=10.")
    print("    More local epochs make more progress per round, but each extra epoch is another noisy step")
    print("    charged to the same budget, so under strong privacy every step gets noisier.")
else:
    print(f"  • E={best_strong} was best at both ε=0.5 and ε=10 in this run.")
gap10 = centralized_auc - m[(best_weak, 10)]
if abs(gap10) < 0.01:
    print(f"  • At ε=10 the best DP-FL model (AUC {m[(best_weak, 10)]:.3f}) matches the non-private centralized model ({centralized_auc:.3f}) within 0.01.")
else:
    print(f"  • At ε=10 the best DP-FL model is {abs(gap10):.3f} AUC {'below' if gap10 > 0 else 'above'} the non-private centralized model.")
print("  • Caveat: one synthetic dataset, linear model, 5 seeds; the ε-guarantee is per hospital for its own patients")
print("    (add/remove-one adjacency, fixed public normaliser and weights; the no-DP column uses the same settings).")


💡 What the sweep shows (this run):
  • E= 1: AUC 0.829 without DP → 0.821 at ε=0.5 (loss 0.008); beats the local-only average (0.804) from ε=0.5 upward
  • E= 5: AUC 0.834 without DP → 0.780 at ε=0.5 (loss 0.053); beats the local-only average (0.804) from ε=1 upward
  • E=10: AUC 0.835 without DP → 0.757 at ε=0.5 (loss 0.078); beats the local-only average (0.804) from ε=1 upward
  • The best number of local epochs depends on ε: E=1 at ε=0.5 but E=10 at ε=10.
    More local epochs make more progress per round, but each extra epoch is another noisy step
    charged to the same budget, so under strong privacy every step gets noisier.
  • At ε=10 the best DP-FL model (AUC 0.835) matches the non-private centralized model (0.834) within 0.01.
  • Caveat: one synthetic dataset, linear model, 5 seeds; the ε-guarantee is per hospital for its own patients
    (add/remove-one adjacency, fixed public normaliser and weights; the no-DP column uses the same settings).


### Privacy-Performance Trade-off Analysis

In [13]:
print("\n📋 Privacy and Security Considerations in Federated Learning:\n")

considerations = [
    {
        'aspect': 'Data Privacy',
        'protection': 'Raw patient data never leaves hospital',
        'limitation': 'Model updates may still leak some information'
    },
    {
        'aspect': 'Model Privacy',
        'protection': 'Secure aggregation hides individual contributions',
        'limitation': 'Global model is shared (potential model inversion)'
    },
    {
        'aspect': 'Communication',
        'protection': 'Only model weights transmitted (efficient)',
        'limitation': 'Multiple rounds needed (communication overhead)'
    },
    {
        'aspect': 'Differential Privacy',
        'protection': 'Can add DP noise to gradients (formal guarantee)',
        'limitation': 'Reduces model accuracy (privacy-utility trade-off)'
    },
    {
        'aspect': 'Compliance',
        'protection': 'Supports data minimization (raw data stays on site)',
        'limitation': 'Requires institutional agreements and governance'
    }
]

for item in considerations:
    print(f"🔹 {item['aspect']}:")
    print(f"   ✓ Protection: {item['protection']}")
    print(f"   ⚠️  Limitation: {item['limitation']}")
    print()

print("="*70)
print("\n💡 Best Practices for Federated Learning in Healthcare:\n")

best_practices = [
    "Combine FL with differential privacy (DP-SGD) for formal privacy guarantees",
    "Use secure aggregation protocols to prevent server from seeing individual updates",
    "Implement secure enclaves (e.g., Intel SGX) for trusted execution",
    "Regularly audit for membership inference and model inversion attacks",
    "Establish clear data governance agreements between institutions",
    "Monitor for Byzantine attacks (malicious institutions sending bad updates)",
    "Use homomorphic encryption for operations on encrypted model updates",
    "Implement client selection strategies to ensure fair representation"
]

for i, practice in enumerate(best_practices, 1):
    print(f"{i}. {practice}")


📋 Privacy and Security Considerations in Federated Learning:

🔹 Data Privacy:
   ✓ Protection: Raw patient data never leaves hospital
   ⚠️  Limitation: Model updates may still leak some information

🔹 Model Privacy:
   ✓ Protection: Secure aggregation hides individual contributions
   ⚠️  Limitation: Global model is shared (potential model inversion)

🔹 Communication:
   ✓ Protection: Only model weights transmitted (efficient)
   ⚠️  Limitation: Multiple rounds needed (communication overhead)

🔹 Differential Privacy:
   ✓ Protection: Can add DP noise to gradients (formal guarantee)
   ⚠️  Limitation: Reduces model accuracy (privacy-utility trade-off)

🔹 Compliance:
   ✓ Protection: Supports data minimization (raw data stays on site)
   ⚠️  Limitation: Requires institutional agreements and governance


💡 Best Practices for Federated Learning in Healthcare:

1. Combine FL with differential privacy (DP-SGD) for formal privacy guarantees
2. Use secure aggregation protocols to prevent ser

---

## Part 6: Clinical Evaluation and Deployment

### Model Performance on Global Test Set

In [14]:
# Detailed evaluation
from sklearn.metrics import confusion_matrix, roc_curve, auc

# Get predictions
fl_predictions = fl_system.predict(global_test_X)
fl_probabilities = fl_system.predict_proba(global_test_X)

# Confusion matrix
cm = confusion_matrix(global_test_y, fl_predictions)
tn, fp, fn, tp = cm.ravel()

# Clinical metrics
sensitivity = tp / (tp + fn)  # Recall / True Positive Rate
specificity = tn / (tn + fp)  # True Negative Rate
ppv = tp / (tp + fp)  # Positive Predictive Value / Precision
npv = tn / (tn + fn)  # Negative Predictive Value

# ROC curve
fpr, tpr, thresholds = roc_curve(global_test_y, fl_probabilities)
roc_auc = auc(fpr, tpr)

print("🏥 Clinical Performance Metrics:\n")
print(f"Sensitivity (Recall):    {sensitivity:.4f}  ← Correctly identified {sensitivity:.1%} of patients with sepsis")
print(f"Specificity:             {specificity:.4f}  ← Correctly identified {specificity:.1%} of healthy patients")
print(f"Positive Predictive Value: {ppv:.4f}  ← {ppv:.1%} of positive predictions are correct")
print(f"Negative Predictive Value: {npv:.4f}  ← {npv:.1%} of negative predictions are correct")
print(f"ROC AUC:                 {roc_auc:.4f}  ← Overall discriminative ability")
print(f"\nConfusion Matrix:")
print(f"  True Negatives:  {tn}")
print(f"  False Positives: {fp}")
print(f"  False Negatives: {fn}")
print(f"  True Positives:  {tp}")

🏥 Clinical Performance Metrics:

Sensitivity (Recall):    0.1690  ← Correctly identified 16.9% of patients with sepsis
Specificity:             0.9770  ← Correctly identified 97.7% of healthy patients
Positive Predictive Value: 0.6000  ← 60.0% of positive predictions are correct
Negative Predictive Value: 0.8521  ← 85.2% of negative predictions are correct
ROC AUC:                 0.8300  ← Overall discriminative ability

Confusion Matrix:
  True Negatives:  340
  False Positives: 8
  False Negatives: 59
  True Positives:  12


In [15]:
# Visualize clinical performance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
           xticklabels=['Predicted Negative', 'Predicted Positive'],
           yticklabels=['Actual Negative', 'Actual Positive'],
           cbar_kws={'label': 'Count'})
axes[0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_ylabel('True Label', fontsize=12)
axes[0].set_xlabel('Predicted Label', fontsize=12)

# ROC curve
axes[1].plot(fpr, tpr, color='darkblue', lw=2,
            label=f'Federated Model (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Random Classifier')
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_ylabel('True Positive Rate (Sensitivity)', fontsize=12)
axes[1].set_title('ROC Curve', fontsize=14, fontweight='bold')
axes[1].legend(loc='lower right')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Clinical Interpretation for Specific Patient

In [16]:
# Example patient prediction
patient_idx = 42
patient_features = global_test_X[patient_idx]
true_label = global_test_y[patient_idx]
predicted_prob = fl_system.predict_proba(patient_features.reshape(1, -1))[0]
predicted_label = fl_system.predict(patient_features.reshape(1, -1))[0]

print("\n" + "="*70)
print("🏥 Clinical Decision Support: Patient Case Example")
print("="*70)
print(f"\nPatient ID: {patient_idx}")
print(f"\nModel Prediction:")
print(f"  Predicted Risk: {predicted_prob:.1%}")
print(f"  Classification: {'POSITIVE - Sepsis likely' if predicted_label == 1 else 'NEGATIVE - Sepsis unlikely'}")
print(f"  Ground Truth: {'Positive' if true_label == 1 else 'Negative'}")
print(f"  Correct Prediction: {'✓ Yes' if predicted_label == true_label else '✗ No'}")

print(f"\n📋 Clinical Recommendation:")
if predicted_prob >= 0.7:
    print("  • HIGH RISK: Bedside review now; start the sepsis pathway per local protocol")
    print("  • (blood cultures, repeat lactate, early antibiotics and fluids if confirmed)")
    print("  • Consider escalation to critical care")
elif predicted_prob >= 0.3:
    print("  • MODERATE RISK: Clinical review within the hour")
    print("  • Increase observation frequency; repeat lactate")
    print("  • Look for a source of infection")
else:
    print("  • LOW RISK: Routine monitoring sufficient")
    print("  • Standard care pathway")
    print("  • Re-assess if vital signs deteriorate")

print("\n⚠️  Important: This model was trained via federated learning across")
print("    10 hospitals WITHOUT sharing patient data, maintaining privacy")
print("    while leveraging collective knowledge.")
print("="*70)


🏥 Clinical Decision Support: Patient Case Example

Patient ID: 42

Model Prediction:
  Predicted Risk: 8.2%
  Classification: NEGATIVE - Sepsis unlikely
  Ground Truth: Positive
  Correct Prediction: ✗ No

📋 Clinical Recommendation:
  • LOW RISK: Routine monitoring sufficient
  • Standard care pathway
  • Re-assess if vital signs deteriorate

⚠️  Important: This model was trained via federated learning across
    10 hospitals WITHOUT sharing patient data, maintaining privacy
    while leveraging collective knowledge.


---

## Summary and Key Takeaways

### What We Learned

1. **Federated Learning Fundamentals**:
   - FedAvg algorithm enables collaborative learning without data sharing
   - Model updates are aggregated using weighted averaging
   - Convergence achieved through multiple rounds of local training + global aggregation

2. **Performance Benefits**:
   - Federated learning can improve on local-only training — in Part 4, check AUC and sensitivity, not just accuracy
   - Accuracy is misleading when sepsis is uncommon: a model that never flags anyone already scores 1 − prevalence
   - Particularly valuable for small hospitals (and, even more, for rare diseases with a handful of cases per site)

3. **Privacy Protections**:
   - Raw patient data never leaves individual hospitals
   - Secure aggregation prevents server from seeing individual updates
   - Per-site differential privacy (clip + noise at each hospital) gives a formal (ε, δ) guarantee; Part 5b measured its cost in AUC across ε and local epochs

4. **Challenges**:
   - Non-IID data across institutions affects convergence
   - Communication overhead (multiple rounds required)
   - Potential for gradient leakage and model inversion attacks
   - Requires governance agreements and institutional coordination

### Clinical Applications

Federated learning is particularly well-suited for:

- **Sepsis and deterioration early warning**: Learn from many case mixes
- **Rare disease prediction**: Pool limited cases across many centers
- **Multi-site clinical trials**: Analyze data without centralization
- **Regional health networks**: Collaborate while respecting local regulations
- **International collaborations**: Overcome cross-border data transfer restrictions
- **Pediatric medicine**: Leverage small sample sizes from children's hospitals

### Best Practices

1. **Technical**:
   - Combine FL with DP-SGD for stronger privacy guarantees
   - Use secure aggregation protocols (e.g., secure multi-party computation)
   - Implement Byzantine-robust aggregation to handle malicious participants
   - Monitor for convergence issues due to non-IID data

2. **Governance**:
   - Establish clear data use agreements between institutions
   - Define ownership of the collaborative model
   - Implement audit trails for model updates and access
   - Ensure compliance with local regulations (HIPAA, GDPR)

3. **Clinical**:
   - Validate federated models against centralized baselines when possible
   - Monitor for distribution shift across participating sites
   - Establish clinical oversight committees
   - Plan for model updates as new institutions join

### Limitations

- **Not a silver bullet**: FL still has privacy vulnerabilities (model inversion, membership inference)
- **Performance gap**: May not match centralized training, especially with extreme non-IID data
- **Communication costs**: Requires reliable networks and multiple rounds
- **Complexity**: More challenging to implement and maintain than centralized training

## Exercises

1. **Rounds and local epochs**: Train a new `FederatedLearning` instance with `train_federated(institutions_data, n_rounds=30, local_epochs=5)`, then with `(15, 1)` and `(15, 20)`. Track global AUC and sensitivity at the 0.5 threshold. Do more rounds fix the low sensitivity (16.9%) or only the ranking?

2. **A threshold for sepsis**: Using `fl_probabilities`, find the threshold that gives about 80% sensitivity and report specificity, PPV and the number of false alarms per true alert (in practice choose it on a validation split of the hospitals' training data, not on the test set). Compare with Marcus's system in Chapter 3 (82% sensitivity, 75% specificity, about 15 false alarms per true alert at 2% prevalence). Why is the ratio so different at this notebook's 17% prevalence?

3. **More extreme non-IID data**: In `generate_hospital_data`, widen the prevalence range `rng.uniform(0.05, 0.25)` to `(0.01, 0.40)` and the age range likewise, then rerun Parts 3–4. How do local-only, federated and centralized AUC change?

4. **A malicious hospital (deeper)**: Make one institution send a corrupted update (for example its weights multiplied by −10) before `aggregate_models`. Measure the global AUC, then replace the weighted mean with a coordinate-wise median and compare. This is the "Byzantine-robust aggregation" best practice in action.

5. **Spending the DP budget**: In Part 5b, call `dp_fedavg` at ε = 1 and E = 1 with `n_rounds` 5 and 30 instead of 15, and with `clip` 0.5 and 2.0. Which combination gives the best AUC for the same guarantee?

6. **Reflection: Marcus's missed alert**: In Chapter 3, Marcus's sepsis model came from three academic centers, and the night nurse had already had 12 alerts that shift when his came. In Part 6, patient 42 had sepsis but scored 8.2% ("routine monitoring"). Would federating with community hospitals like Marcus's have helped, and how (case mix, calibration to local prevalence)? Which threshold would you pick for a ward whose nurses already get a dozen alerts a shift?

---

### Further Reading

- McMahan et al. (2017): "Communication-Efficient Learning of Deep Networks from Decentralized Data"
- Bonawitz et al. (2019): "Towards Federated Learning at Scale: System Design"
- Li et al. (2020): "Federated Learning on Non-IID Data Silos"
- Kaissis et al. (2020): "Secure, privacy-preserving and federated machine learning in medical imaging"
- Rieke et al. (2020): "The future of digital health with federated learning"

### Next Steps

In **Notebook 11.3**, we'll explore **Adversarial Robustness**, examining how to defend AI models against malicious attacks that attempt to manipulate predictions through carefully crafted inputs.

---

**End of Notebook 11.2**